# systeme de recommandation k-nn pour biens immobiliers

**objectif** : recommander des biens similaires a partir des annonces actuelles bienici.

**methode** : k-nearest neighbors (k-nn) implemente from scratch.

**reference** : joel grus, "data science from scratch", chapitre 12 (k-nearest neighbors). [cite: 121-130]

---

## principe du k-nn

le k-nn est un algorithme de machine learning simple mais efficace pour la recommandation :

1. **distance** : on calcule la distance euclidienne entre le bien recherche et tous les biens du catalogue
2. **voisinage** : on selectionne les k biens les plus proches
3. **recommandation** : ces k biens sont les recommandations

**formule de distance euclidienne** : [cite: 123]

$$d(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$$

ou $p$ et $q$ sont deux vecteurs de features, et $n$ est le nombre de dimensions.

In [1]:
# imports
import csv
import math
from pathlib import Path

## 1. fonctions de base from scratch

implementation manuelle de toutes les fonctions necessaires au k-nn.

In [2]:
def distance_euclidienne(v1, v2):
    """
    calcule la distance euclidienne entre deux vecteurs.
    [cite: 123]
    """
    if len(v1) != len(v2):
        raise ValueError("les vecteurs doivent avoir la meme dimension")
    
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(v1, v2)))

In [3]:
def normaliser_features(data, stats=None):
    """
    normalise les features avec min-max scaling : (x - min) / (max - min).
    permet de mettre toutes les variables sur la meme echelle. [cite: 124]
    
    retourne :
    - data normalisee
    - stats (pour denormaliser plus tard si besoin)
    """
    if not data:
        return [], None
    
    num_features = len(data[0])
    
    # calcul des stats si pas fournies
    if stats is None:
        stats = []
        for i in range(num_features):
            values = [row[i] for row in data]
            min_val = min(values)
            max_val = max(values)
            stats.append((min_val, max_val))
    
    # normalisation
    data_norm = []
    for row in data:
        row_norm = []
        for i, val in enumerate(row):
            min_val, max_val = stats[i]
            if max_val - min_val == 0:
                row_norm.append(0.0)  # si constante, on met 0
            else:
                row_norm.append((val - min_val) / (max_val - min_val))
        data_norm.append(row_norm)
    
    return data_norm, stats

In [4]:
def knn_similaires(bien_recherche, catalogue, k=5, normaliser=True):
    """
    trouve les k biens les plus similaires au bien recherche.
    [cite: 125-127]
    
    args:
        bien_recherche : vecteur de features du bien cible
        catalogue : liste de tuples (vecteur_features, annonce_complete)
        k : nombre de voisins a retourner
        normaliser : normaliser les features avant calcul distance
    
    retourne:
        liste des k annonces les plus similaires avec leur distance
    """
    # extraction des vecteurs
    vecteurs = [item[0] for item in catalogue]
    
    # normalisation si demandee
    if normaliser:
        tous_vecteurs = [bien_recherche] + vecteurs
        vecteurs_norm, stats = normaliser_features(tous_vecteurs)
        bien_norm = vecteurs_norm[0]
        catalogue_norm = vecteurs_norm[1:]
    else:
        bien_norm = bien_recherche
        catalogue_norm = vecteurs
    
    # calcul des distances
    distances = []
    for i, vecteur_norm in enumerate(catalogue_norm):
        dist = distance_euclidienne(bien_norm, vecteur_norm)
        distances.append((dist, catalogue[i][1]))  # (distance, annonce)
    
    # tri par distance croissante et selection des k premiers
    distances.sort(key=lambda x: x[0])
    
    return distances[:k]

## 2. chargement des donnees bienici

on charge les annonces actuelles du marche toulonnais.

In [5]:
# chargement des annonces
annonces = []
csv_path = Path('../data/annonces_bienici_clean.csv')

with open(csv_path, 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f, delimiter=',')
    for row in reader:
        try:
            annonce = {
                'id': row['id_annonce'],
                'url': row['url'],
                'type': row['type_bien'],
                'prix': float(row['prix']),
                'surface': float(row['surface_m2']),
                'prix_m2': float(row['prix_m2']) if row['prix_m2'] else None,
                'pieces': int(float(row['pieces'])) if row['pieces'] else 0,
                'chambres': int(float(row['chambres'])) if row['chambres'] else 0,
                'code_postal': row['code_postal'],
                'balcon': row['balcon'] == '1',
                'terrasse': row['terrasse'] == '1',
                'parking': int(float(row['nb_parkings'])) if row['nb_parkings'] else 0,
                'ascenseur': row['ascenseur'] == '1'
            }
            annonces.append(annonce)
        except (ValueError, TypeError, KeyError) as e:
            continue

print(f'{len(annonces)} annonces chargees')
print(f'exemple d\'annonce :')
print(annonces[0])

1332 annonces chargees
exemple d'annonce :
{'id': 'ag311656-510682583', 'url': 'https://www.bienici.com/annonce/vente/toulon/flat/ag311656-510682583', 'type': 'Appartement', 'prix': 42000.0, 'surface': 18.0, 'prix_m2': 2333.33, 'pieces': 1, 'chambres': 0, 'code_postal': '83000', 'balcon': False, 'terrasse': False, 'parking': 0, 'ascenseur': False}


## 3. vectorisation des annonces

chaque annonce est transformee en vecteur de 13 dimensions pour le k-nn.

In [6]:
def annonce_vers_vecteur(annonce):
    """
    convertit une annonce en vecteur de features pour k-nn.
    
    features selectionnees (13 dimensions) :
    1. surface (m2)
    2. prix (euros)
    3. prix au m2
    4. nombre de pieces
    5. nombre de chambres
    6. type (0=appartement, 1=maison)
    7-9. code postal (one-hot : 83000, 83100, 83200)
    10. balcon (0/1)
    11. terrasse (0/1)
    12. nombre de parkings
    13. ascenseur (0/1)
    """
    # encodage du type
    type_num = 1.0 if annonce['type'] == 'Maison' else 0.0
    
    # encodage one-hot du code postal
    is_83000 = 1.0 if annonce['code_postal'] == '83000' else 0.0
    is_83100 = 1.0 if annonce['code_postal'] == '83100' else 0.0
    is_83200 = 1.0 if annonce['code_postal'] == '83200' else 0.0
    
    # calcul prix_m2 si manquant
    prix_m2 = annonce['prix_m2'] if annonce['prix_m2'] else annonce['prix'] / annonce['surface']
    
    return [
        annonce['surface'],
        annonce['prix'],
        prix_m2,
        annonce['pieces'],
        annonce['chambres'],
        type_num,
        is_83000,
        is_83100,
        is_83200,
        float(annonce['balcon']),
        float(annonce['terrasse']),
        annonce['parking'],
        float(annonce['ascenseur'])
    ]

In [7]:
# creation du catalogue avec vecteurs
catalogue = [(annonce_vers_vecteur(a), a) for a in annonces]

print(f'catalogue cree avec {len(catalogue)} biens')
print(f'\nexemple de vecteur (13 dimensions) :')
print(catalogue[0][0])
print(f'\ncorrespondant a l\'annonce :')
print(f"type: {catalogue[0][1]['type']}, surface: {catalogue[0][1]['surface']}m2, prix: {catalogue[0][1]['prix']:,.0f} euros")

catalogue cree avec 1332 biens

exemple de vecteur (13 dimensions) :
[18.0, 42000.0, 2333.33, 1, 0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0, 0.0]

correspondant a l'annonce :
type: Appartement, surface: 18.0m2, prix: 42,000 euros


## 4. fonctions metier pour niddouillet

fonctions adaptees aux besoins de l'agence.

In [8]:
def recommander_annonces(criteres, annonces, catalogue, k=5):
    """
    recommande k annonces similaires aux criteres de recherche.
    
    args:
        criteres : dict avec les criteres du client
        annonces : liste complete des annonces
        catalogue : catalogue vectorise
        k : nombre de recommandations
    
    retourne:
        liste des k meilleures recommandations
    """
    # creation du vecteur de recherche
    vecteur_recherche = annonce_vers_vecteur(criteres)
    
    # recherche des k plus proches voisins
    resultats = knn_similaires(vecteur_recherche, catalogue, k=k)
    
    return resultats

In [9]:
def afficher_recommandations(resultats):
    """
    affiche les recommandations de maniere lisible.
    """
    print(f'\n=== {len(resultats)} recommandations ===')
    print()
    
    for i, (distance, annonce) in enumerate(resultats, 1):
        print(f'{i}. {annonce["type"]} - {annonce["surface"]}m2 - {annonce["pieces"]} pieces')
        print(f'   prix : {annonce["prix"]:,.0f} euros ({annonce["prix"] / annonce["surface"]:,.0f} euros/m2)')
        print(f'   quartier : {annonce["code_postal"]}')
        
        # equipements
        equip = []
        if annonce['balcon']:
            equip.append('balcon')
        if annonce['terrasse']:
            equip.append('terrasse')
        if annonce['parking'] > 0:
            equip.append(f'{annonce["parking"]} parking(s)')
        if annonce['ascenseur']:
            equip.append('ascenseur')
        
        if equip:
            print(f'   equipements : {", ".join(equip)}')
        
        print(f'   similarite : {(1 - min(distance, 1)) * 100:.1f}%')
        print(f'   url : {annonce["url"]}')
        print()

In [10]:
def analyser_similarite(annonce_ref, annonce_compare):
    """
    explique pourquoi deux annonces sont similaires.
    """
    print('\n=== analyse de similarite ===')
    print()
    
    # comparaison surface
    diff_surface = abs(annonce_ref['surface'] - annonce_compare['surface'])
    pct_surface = (diff_surface / annonce_ref['surface']) * 100
    print(f'surface : {annonce_ref["surface"]}m2 vs {annonce_compare["surface"]}m2 ({pct_surface:.1f}% ecart)')
    
    # comparaison prix
    diff_prix = abs(annonce_ref['prix'] - annonce_compare['prix'])
    pct_prix = (diff_prix / annonce_ref['prix']) * 100
    print(f'prix : {annonce_ref["prix"]:,.0f} vs {annonce_compare["prix"]:,.0f} euros ({pct_prix:.1f}% ecart)')
    
    # comparaison type
    meme_type = annonce_ref['type'] == annonce_compare['type']
    print(f'type : {"identique" if meme_type else "different"} ({annonce_ref["type"]} vs {annonce_compare["type"]})')
    
    # comparaison pieces
    meme_pieces = annonce_ref['pieces'] == annonce_compare['pieces']
    print(f'pieces : {"identique" if meme_pieces else "different"} ({annonce_ref["pieces"]} vs {annonce_compare["pieces"]})')
    
    # comparaison quartier
    meme_quartier = annonce_ref['code_postal'] == annonce_compare['code_postal']
    print(f'quartier : {"identique" if meme_quartier else "different"} ({annonce_ref["code_postal"]} vs {annonce_compare["code_postal"]})')
    
    print()

In [11]:
def statistiques_recommandations(resultats):
    """
    calcule des statistiques sur les recommandations.
    """
    annonces_reco = [r[1] for r in resultats]
    
    print('\n=== statistiques des recommandations ===')
    print()
    
    # prix
    prix_list = [a['prix'] for a in annonces_reco]
    print(f'prix moyen : {sum(prix_list) / len(prix_list):,.0f} euros')
    print(f'prix min : {min(prix_list):,.0f} euros')
    print(f'prix max : {max(prix_list):,.0f} euros')
    print()
    
    # surface
    surface_list = [a['surface'] for a in annonces_reco]
    print(f'surface moyenne : {sum(surface_list) / len(surface_list):.1f}m2')
    print(f'surface min : {min(surface_list):.1f}m2')
    print(f'surface max : {max(surface_list):.1f}m2')
    print()
    
    # types
    types_count = {}
    for a in annonces_reco:
        t = a['type']
        types_count[t] = types_count.get(t, 0) + 1
    
    print('repartition par type :')
    for t, count in types_count.items():
        print(f'  {t} : {count} ({count / len(annonces_reco) * 100:.0f}%)')
    print()

## 5. exemple d'utilisation : recherche personnalisee

demonstration avec des criteres modifiables.

In [12]:
# criteres de recherche (modifiables)
criteres_recherche = {
    'type': 'Appartement',
    'surface': 70,
    'prix': 250000,
    'prix_m2': 3571,
    'pieces': 3,
    'chambres': 2,
    'code_postal': '83000',
    'balcon': True,
    'terrasse': False,
    'parking': 1,
    'ascenseur': True
}

print('=== criteres de recherche ===')
print()
print(f"{criteres_recherche['type']} {criteres_recherche['surface']}m2 - {criteres_recherche['pieces']} pieces")
print(f"budget : {criteres_recherche['prix']:,.0f} euros")
print(f"quartier : {criteres_recherche['code_postal']}")
print()

# recherche des 5 biens les plus similaires
recommandations = recommander_annonces(criteres_recherche, annonces, catalogue, k=5)

# affichage
afficher_recommandations(recommandations)

=== criteres de recherche ===

Appartement 70m2 - 3 pieces
budget : 250,000 euros
quartier : 83000


=== 5 recommandations ===

1. Appartement - 83.0m2 - 4 pieces
   prix : 259,000 euros (3,120 euros/m2)
   quartier : 83000
   equipements : balcon, ascenseur
   similarite : 89.2%
   url : https://www.bienici.com/annonce/vente/toulon/flat/ag942352-517769198

2. Appartement - 92.0m2 - 4 pieces
   prix : 277,000 euros (3,011 euros/m2)
   quartier : 83000
   equipements : balcon, ascenseur
   similarite : 80.7%
   url : https://www.bienici.com/annonce/vente/toulon/flat/ag311656-505665329

3. Appartement - 82.0m2 - 4 pieces
   prix : 199,500 euros (2,433 euros/m2)
   quartier : 83000
   equipements : balcon, 1 parking(s), ascenseur
   similarite : 75.6%
   url : https://www.bienici.com/annonce/vente/toulon/flat/ag920856-500617394

4. Appartement - 72.0m2 - 4 pieces
   prix : 187,000 euros (2,597 euros/m2)
   quartier : 83000
   equipements : balcon, ascenseur
   similarite : 75.6%
   url : 

In [13]:
# analyse de la premiere recommandation
if recommandations:
    analyser_similarite(criteres_recherche, recommandations[0][1])


=== analyse de similarite ===

surface : 70m2 vs 83.0m2 (18.6% ecart)
prix : 250,000 vs 259,000 euros (3.6% ecart)
type : identique (Appartement vs Appartement)
pieces : different (3 vs 4)
quartier : identique (83000 vs 83000)



In [14]:
# statistiques sur les recommandations
statistiques_recommandations(recommandations)


=== statistiques des recommandations ===

prix moyen : 223,500 euros
prix min : 187,000 euros
prix max : 277,000 euros

surface moyenne : 82.2m2
surface min : 72.0m2
surface max : 92.0m2

repartition par type :
  Appartement : 5 (100%)



## 6. cas d'usage reel : primo-accedant niddouillet

simulation d'une recherche pour un client type de l'agence.

In [15]:
# profil client niddouillet
client_primo = {
    'type': 'Appartement',
    'surface': 55,
    'prix': 175000,  # budget max 450k, vise 175k pour premier achat
    'prix_m2': 3182,
    'pieces': 3,  # T3 ideal pour jeune couple
    'chambres': 2,
    'code_postal': '83100',  # toulon centre
    'balcon': True,  # souhaite exterieur
    'terrasse': False,
    'parking': 1,  # indispensable
    'ascenseur': False  # pas prioritaire pour reduire charges
}

print('=== client primo-accedant niddouillet ===')
print()
print(f"profil : jeune couple, premier achat")
print(f"recherche : {client_primo['type']} T{client_primo['pieces']} environ {client_primo['surface']}m2")
print(f"budget : {client_primo['prix']:,.0f} euros")
print(f"criteres : balcon + parking indispensable")
print()

# recherche
reco_client = recommander_annonces(client_primo, annonces, catalogue, k=10)

# affichage top 5
afficher_recommandations(reco_client[:5])

=== client primo-accedant niddouillet ===

profil : jeune couple, premier achat
recherche : Appartement T3 environ 55m2
budget : 175,000 euros
criteres : balcon + parking indispensable


=== 5 recommandations ===

1. Appartement - 69.0m2 - 3 pieces
   prix : 179,000 euros (2,594 euros/m2)
   quartier : 83100
   equipements : balcon
   similarite : 84.1%
   url : https://www.bienici.com/annonce/vente/toulon/flat/ag311656-516895547

2. Appartement - 123.0m2 - 5 pieces
   prix : 349,000 euros (2,837 euros/m2)
   quartier : 83100
   equipements : balcon
   similarite : 43.7%
   url : https://www.bienici.com/annonce/vente/toulon/flat/ag835481-462277949

3. Appartement - 60.0m2 - 3 pieces
   prix : 185,000 euros (3,083 euros/m2)
   quartier : 83100
   similarite : 0.0%
   url : https://www.bienici.com/annonce/vente/toulon/flat/iad-france-1991752

4. Appartement - 57.0m2 - 3 pieces
   prix : 169,000 euros (2,965 euros/m2)
   quartier : 83100
   similarite : 0.0%
   url : https://www.bienici.c

In [16]:
# analyse des recommandations pour le client
print('\n=== analyse des opportunites pour le client ===')
print()

for i, (dist, annonce) in enumerate(reco_client[:5], 1):
    ecart_prix = annonce['prix'] - client_primo['prix']
    ecart_surface = annonce['surface'] - client_primo['surface']
    
    print(f'{i}. {annonce["id"]}')
    print(f'   prix : {ecart_prix:+,.0f} euros par rapport au budget')
    print(f'   surface : {ecart_surface:+.0f}m2 par rapport a la recherche')
    
    # opportunite ?
    if ecart_prix <= 0 and annonce['surface'] >= client_primo['surface'] * 0.9:
        print('   => OPPORTUNITE : prix interessant pour surface equivalente')
    elif ecart_prix <= 10000 and annonce['surface'] > client_primo['surface']:
        print('   => BON COMPROMIS : leger surcout pour plus de surface')
    
    print()


=== analyse des opportunites pour le client ===

1. ag311656-516895547
   prix : +4,000 euros par rapport au budget
   surface : +14m2 par rapport a la recherche
   => BON COMPROMIS : leger surcout pour plus de surface

2. ag835481-462277949
   prix : +174,000 euros par rapport au budget
   surface : +68m2 par rapport a la recherche

3. iad-france-1991752
   prix : +10,000 euros par rapport au budget
   surface : +5m2 par rapport a la recherche
   => BON COMPROMIS : leger surcout pour plus de surface

4. hektor-ageris-235299
   prix : -6,000 euros par rapport au budget
   surface : +2m2 par rapport a la recherche
   => OPPORTUNITE : prix interessant pour surface equivalente

5. nestenn-1-99939437658
   prix : +30,000 euros par rapport au budget
   surface : +6m2 par rapport a la recherche



## 7. test : recherche maisons

le k-nn fonctionne aussi pour les maisons.

In [17]:
# recherche maison
criteres_maison = {
    'type': 'Maison',
    'surface': 100,
    'prix': 350000,
    'prix_m2': 3500,
    'pieces': 5,
    'chambres': 3,
    'code_postal': '83200',  # quartiers peripheriques
    'balcon': False,
    'terrasse': True,  # terrasse importante pour maison
    'parking': 2,  # 2 places
    'ascenseur': False
}

print('=== recherche maison familiale ===')
print()
print(f"{criteres_maison['type']} {criteres_maison['surface']}m2 - {criteres_maison['pieces']} pieces")
print(f"budget : {criteres_maison['prix']:,.0f} euros")
print()

reco_maison = recommander_annonces(criteres_maison, annonces, catalogue, k=5)
afficher_recommandations(reco_maison)

=== recherche maison familiale ===

Maison 100m2 - 5 pieces
budget : 350,000 euros


=== 5 recommandations ===

1. Maison - 91.0m2 - 5 pieces
   prix : 350,000 euros (3,846 euros/m2)
   quartier : 83200
   equipements : terrasse, 2 parking(s)
   similarite : 94.2%
   url : https://www.bienici.com/annonce/vente/toulon/house/ag922880-512198780

2. Maison - 91.0m2 - 4 pieces
   prix : 370,000 euros (4,066 euros/m2)
   quartier : 83200
   equipements : terrasse
   similarite : 87.3%
   url : https://www.bienici.com/annonce/vente/toulon/house/iad-france-2020440

3. Maison - 130.0m2 - 5 pieces
   prix : 379,000 euros (2,915 euros/m2)
   quartier : 83200
   equipements : terrasse, 2 parking(s)
   similarite : 84.5%
   url : https://www.bienici.com/annonce/vente/toulon/house/safti-1-1522784

4. Maison - 90.0m2 - 4 pieces
   prix : 390,000 euros (4,333 euros/m2)
   quartier : 83200
   equipements : terrasse, 1 parking(s)
   similarite : 83.3%
   url : https://www.bienici.com/annonce/vente/toulo

## conclusion

le systeme k-nn permet de recommander des biens similaires en se basant sur 13 criteres :

- **caracteristiques physiques** : surface, prix, prix/m2, pieces, chambres, type
- **localisation** : code postal (one-hot encoding)
- **equipements** : balcon, terrasse, parking, ascenseur

**avantages** :
- simple a comprendre et expliquer aux clients
- pas de phase d'entrainement (modele non parametrique)
- s'adapte automatiquement aux nouvelles annonces
- from scratch : pas de dependance externe

**limites** :
- calcul de distance pour chaque recherche (peut etre lent avec beaucoup d'annonces)
- toutes les features ont le meme poids (apres normalisation)
- pas de prise en compte de l'historique des preferences client

**ameliorations possibles** :
- ponderation des features selon leur importance
- distance de manhattan pour features categorielles
- clustering prealable pour accelerer les recherches